# Graphs: BFS, DFS, Topological Sort, Union-Find and Shortest Paths

Notebook for **Chapter 8** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 Graph Representations: Adjacency List, Matrix and Edge List

In [ ]:
from collections import defaultdict

def build_graph(n, edges, directed=False):
    graph = [[] for _ in range(n)]      # one empty list per node 0..n-1
    for a, b in edges:
        graph[a].append(b)              # edge a -> b
        if not directed:
            graph[b].append(a)          # undirected: also b -> a
    return graph

edges = [(0, 1), (0, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 5)]
print(build_graph(6, edges))
print(build_graph(6, edges, directed=True))

# the same graph as a dict of sets (handy when nodes are names, not numbers)
names = defaultdict(set)
for a, b in [("ann", "bo"), ("bo", "cy"), ("ann", "cy")]:
    names[a].add(b)
    names[b].add(a)
print({k: sorted(v) for k, v in sorted(names.items())})

[[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]
[[1, 2], [3], [3, 4], [5], [5], []]
{'ann': ['bo', 'cy'], 'bo': ['ann', 'cy'], 'cy': ['ann', 'bo']}


In [ ]:
def degrees(n, edges):
    deg = [0] * n
    for a, b in edges:
        deg[a] += 1
        deg[b] += 1
    return deg

edges = [(0, 1), (0, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 5)]
deg = degrees(6, edges)
print(deg)
best = max(range(6), key=lambda v: deg[v])
print("most connected:", best)
assert sum(deg) == 2 * len(edges)      # handshake rule: degrees add up to twice the edges
assert degrees(4, [(0, 1), (0, 2), (1, 3)]) == [2, 2, 1, 1]

[2, 2, 3, 3, 2, 2]
most connected: 2


In [ ]:
def star_center(edges):
    a, b = edges[0]
    c, d = edges[1]
    return a if a in (c, d) else b

assert star_center([(1, 2), (2, 3), (4, 2)]) == 2
assert star_center([(1, 2), (5, 1), (1, 3), (1, 4)]) == 1
print(star_center([(1, 2), (2, 3), (4, 2)]))
print(star_center([(1, 2), (5, 1), (1, 3), (1, 4)]))

2
1


## 8.2 Modelling: Grids, Word Ladders and Other Hidden Graphs

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]     # up, down, left, right

def grid_neighbours(grid, r, c):
    rows, cols = len(grid), len(grid[0])
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:   # 0 = open, 1 = wall
            yield (nr, nc)

grid = [[0, 0, 1],
        [0, 1, 0],
        [0, 0, 0]]
print(list(grid_neighbours(grid, 1, 0)))
print(list(grid_neighbours(grid, 0, 0)))
print(list(grid_neighbours(grid, 2, 2)))

[(0, 0), (2, 0)]
[(1, 0), (0, 1)]
[(1, 2), (2, 1)]


In [ ]:
from collections import defaultdict

def build_buckets(words):
    buckets = defaultdict(list)
    for w in words:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    return buckets

def word_neighbours(word, buckets):
    result = set()
    for i in range(len(word)):
        for other in buckets[word[:i] + "*" + word[i + 1:]]:
            if other != word:
                result.add(other)
    return sorted(result)

words = ["hit", "hot", "dot", "dog", "lot", "log", "cog"]
buckets = build_buckets(words)
print(sorted(buckets["h*t"]), sorted(buckets["*ot"]), sorted(buckets["do*"]))
print(word_neighbours("hot", buckets))
print(word_neighbours("dog", buckets))

['hit', 'hot'] ['dot', 'hot', 'lot'] ['dog', 'dot']
['dot', 'hit', 'lot']
['cog', 'dot', 'log']


In [ ]:
def move_counts(grid):
    rows, cols = len(grid), len(grid[0])
    result = [[-1] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                continue
            count = 0
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
                    count += 1
            result[r][c] = count
    return result

print(move_counts([[0, 0], [0, 1]]))
print(move_counts([[0, 0, 1], [0, 1, 0], [0, 0, 0]]))
assert move_counts([[0]]) == [[0]]

[[2, 1], [1, -1]]
[[2, 1, -1], [2, -1, 1], [2, 2, 2]]


## 8.3 BFS: Shortest Paths in Unweighted Graphs

In [ ]:
from collections import deque

def bfs(graph, start):
    dist = {start: 0}                   # dist doubles as the visited set
    queue = deque([start])
    while queue:
        node = queue.popleft()          # oldest item first
        for nxt in graph[node]:
            if nxt not in dist:         # first arrival = shortest
                dist[nxt] = dist[node] + 1
                queue.append(nxt)       # mark when pushing, not when popping
    return dist

graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]
print(bfs(graph, 0))

{0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3}


In [ ]:
from collections import deque

def shortest_path(graph, start, target):
    parent = {start: None}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == target:
            break
        for nxt in graph[node]:
            if nxt not in parent:
                parent[nxt] = node
                queue.append(nxt)
    if target not in parent:
        return []
    path = []
    while target is not None:           # walk back to the start
        path.append(target)
        target = parent[target]
    return path[::-1]

graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4], []]   # node 6 is isolated
print(shortest_path(graph, 0, 5))
print(shortest_path(graph, 4, 4))
print(shortest_path(graph, 0, 6))
assert shortest_path(graph, 0, 5) == [0, 1, 3, 5]

[0, 1, 3, 5]
[4]
[]


In [ ]:
from collections import deque

def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    queue, fresh = deque(), 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                queue.append((r, c))
            elif grid[r][c] == 1:
                fresh += 1
    minutes = 0
    while queue and fresh:
        for _ in range(len(queue)):     # process exactly one layer = one minute
            r, c = queue.popleft()
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    queue.append((nr, nc))
        minutes += 1
    return minutes if fresh == 0 else -1

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))
print(oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))
print(oranges_rotting([[0, 2]]))

4
-1
0


In [ ]:
from collections import deque, defaultdict

def ladder_length(begin, end, words):
    if end not in words:
        return 0
    buckets = defaultdict(list)
    for w in set(words) | {begin}:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    seen = {begin}
    queue = deque([(begin, 1)])
    while queue:
        word, length = queue.popleft()
        if word == end:
            return length
        for i in range(len(word)):
            for nxt in buckets[word[:i] + "*" + word[i + 1:]]:
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, length + 1))
    return 0

words = ["hot", "dot", "dog", "lot", "log", "cog"]
assert ladder_length("hit", "cog", words) == 5
assert ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"]) == 0
print(ladder_length("hit", "cog", words))
print(ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"]))

5
0


## 8.4 DFS: Connected Components, Cycles and Islands

In [ ]:
def dfs(graph, node, seen, order):
    seen.add(node)
    order.append(node)                  # "visit" the node
    for nxt in graph[node]:
        if nxt not in seen:
            dfs(graph, nxt, seen, order)

def count_components(graph):
    seen, count = set(), 0
    for start in range(len(graph)):     # outer loop catches every island
        if start not in seen:
            dfs(graph, start, seen, [])
            count += 1
    return count

graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]
order, seen = [], set()
dfs(graph, 0, seen, order)
print(order)
two_groups = [[1], [0], [3], [2], []]            # {0,1}, {2,3}, {4}
print(count_components(two_groups))

[0, 1, 3, 2, 4, 5]
3


In [ ]:
def dfs_iterative(graph, start):
    seen = {start}
    stack = [start]
    order = []
    while stack:
        node = stack.pop()              # newest item first
        order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                stack.append(nxt)
    return order

graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]
print(dfs_iterative(graph, 0))

[0, 2, 4, 5, 3, 1]


In [ ]:
def num_islands(grid):
    grid = [list(row) for row in grid]  # copy so the caller's data is unchanged
    rows, cols = len(grid), len(grid[0]) if grid else 0
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "1":
                continue
            count += 1
            stack = [(r, c)]
            grid[r][c] = "0"            # sink on push
            while stack:
                cr, cc = stack.pop()
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                        grid[nr][nc] = "0"
                        stack.append((nr, nc))
    return count

g = ["11000", "11000", "00100", "00011", "00011"]
assert num_islands(g) == 3
assert num_islands(["000"]) == 0
assert num_islands([]) == 0
print(num_islands(g), num_islands(["111", "111"]), num_islands([]))

3 1 0


In [ ]:
def has_cycle_undirected(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen = set()

    def dfs(node, parent):
        seen.add(node)
        for nxt in graph[node]:
            if nxt == parent:
                continue                # the edge we just used
            if nxt in seen or dfs(nxt, node):
                return True
        return False

    return any(dfs(s, -1) for s in range(n) if s not in seen)

assert has_cycle_undirected(3, [(0, 1), (1, 2), (2, 0)]) is True
assert has_cycle_undirected(3, [(0, 1), (1, 2)]) is False
assert has_cycle_undirected(5, [(0, 1), (2, 3), (3, 4), (4, 2)]) is True
print(has_cycle_undirected(3, [(0, 1), (1, 2), (2, 0)]), has_cycle_undirected(3, [(0, 1), (1, 2)]))

True False


In [ ]:
def has_cycle_directed(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state = [0] * n                     # 0 new, 1 on current path, 2 finished

    def dfs(node):
        state[node] = 1
        for nxt in graph[node]:
            if state[nxt] == 1:
                return True             # back to a node on our path
            if state[nxt] == 0 and dfs(nxt):
                return True
        state[node] = 2
        return False

    return any(state[s] == 0 and dfs(s) for s in range(n))

assert has_cycle_directed(3, [(0, 1), (1, 2), (2, 0)]) is True
assert has_cycle_directed(3, [(0, 1), (0, 2), (1, 2)]) is False
assert has_cycle_directed(2, [(1, 1)]) is True
print(has_cycle_directed(3, [(0, 1), (1, 2), (2, 0)]), has_cycle_directed(3, [(0, 1), (0, 2), (1, 2)]))

True False


## 8.5 Topological Sort: Kahn and DFS, Course Schedule

In [ ]:
from collections import deque

def topo_kahn(n, edges):
    graph = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in edges:                  # a must come before b
        graph[a].append(b)
        indeg[b] += 1
    queue = deque(v for v in range(n) if indeg[v] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nxt in graph[node]:
            indeg[nxt] -= 1             # one prerequisite is done
            if indeg[nxt] == 0:
                queue.append(nxt)
    return order if len(order) == n else []     # fewer than n means a cycle

edges = [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]
print(topo_kahn(6, edges))
print(topo_kahn(3, [(0, 1), (1, 2), (2, 0)]))

[4, 5, 2, 0, 3, 1]
[]


In [ ]:
def topo_dfs(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    state = [0] * n                     # 0 new, 1 on path, 2 done
    post = []

    def dfs(node):
        state[node] = 1
        for nxt in graph[node]:
            if state[nxt] == 1:
                return False            # cycle
            if state[nxt] == 0 and not dfs(nxt):
                return False
        state[node] = 2
        post.append(node)               # all descendants already added
        return True

    for s in range(n):
        if state[s] == 0 and not dfs(s):
            return []
    return post[::-1]

edges = [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]
order = topo_dfs(6, edges)
print(order)
pos = {v: i for i, v in enumerate(order)}
assert all(pos[a] < pos[b] for a, b in edges)   # every arrow goes left to right
print(topo_dfs(2, [(0, 1), (1, 0)]))

[5, 4, 2, 3, 1, 0]
[]


In [ ]:
from collections import deque

def can_finish(n, prereqs):
    graph = [[] for _ in range(n)]
    indeg = [0] * n
    for course, before in prereqs:      # note the direction: before -> course
        graph[before].append(course)
        indeg[course] += 1
    queue = deque(v for v in range(n) if indeg[v] == 0)
    done = 0
    while queue:
        node = queue.popleft()
        done += 1
        for nxt in graph[node]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                queue.append(nxt)
    return done == n

assert can_finish(2, [[1, 0]]) is True
assert can_finish(2, [[1, 0], [0, 1]]) is False
assert can_finish(3, []) is True
assert can_finish(1, [[0, 0]]) is False
print(can_finish(2, [[1, 0]]), can_finish(2, [[1, 0], [0, 1]]), can_finish(3, []), can_finish(1, [[0, 0]]))

True False True False


In [ ]:
from collections import deque

def min_semesters(n, edges):
    graph = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in edges:
        graph[a].append(b)
        indeg[b] += 1
    queue = deque(v for v in range(n) if indeg[v] == 0)
    semesters, taken = 0, 0
    while queue:
        semesters += 1
        for _ in range(len(queue)):     # one whole layer = one semester
            node = queue.popleft()
            taken += 1
            for nxt in graph[node]:
                indeg[nxt] -= 1
                if indeg[nxt] == 0:
                    queue.append(nxt)
    return semesters if taken == n else -1

edges = [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]
assert min_semesters(6, edges) == 4
assert min_semesters(3, []) == 1
assert min_semesters(2, [(0, 1), (1, 0)]) == -1
print(min_semesters(6, edges), min_semesters(3, []), min_semesters(2, [(0, 1), (1, 0)]))

4 1 -1


In [ ]:
from collections import deque

def alien_order(words):
    graph = {c: set() for w in words for c in w}
    indeg = {c: 0 for c in graph}
    for w1, w2 in zip(words, words[1:]):
        if len(w1) > len(w2) and w1.startswith(w2):
            return ""                   # "abc" before "ab" is impossible
        for a, b in zip(w1, w2):
            if a != b:
                if b not in graph[a]:
                    graph[a].add(b)
                    indeg[b] += 1
                break                   # only the first difference counts
    queue = deque(sorted(c for c in indeg if indeg[c] == 0))
    out = []
    while queue:
        c = queue.popleft()
        out.append(c)
        for nxt in sorted(graph[c]):
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                queue.append(nxt)
    return "".join(out) if len(out) == len(graph) else ""

assert alien_order(["wrt", "wrf", "er", "ett", "rftt"]) == "wertf"
assert alien_order(["z", "x", "z"]) == ""
assert alien_order(["abc", "ab"]) == ""
print(alien_order(["wrt", "wrf", "er", "ett", "rftt"]))
print(repr(alien_order(["z", "x", "z"])), repr(alien_order(["abc", "ab"])))

wertf
'' ''


## 8.6 Union-Find (Disjoint Set Union)

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))    # everyone is their own leader
        self.rank = [0] * n             # upper bound on tree height
        self.groups = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving (compression)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                # already in the same group
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra             # ra is now the taller tree
        self.parent[rb] = ra            # hang the shorter tree under the taller
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        self.groups -= 1
        return True

d = DSU(6)
for a, b in [(0, 1), (2, 3), (1, 3), (4, 5), (3, 0), (2, 5)]:
    print((a, b), d.union(a, b), "parent", d.parent, "groups", d.groups)

(0, 1) True parent [0, 0, 2, 3, 4, 5] groups 5
(2, 3) True parent [0, 0, 2, 2, 4, 5] groups 4
(1, 3) True parent [0, 0, 0, 2, 4, 5] groups 3
(4, 5) True parent [0, 0, 0, 2, 4, 4] groups 2
(3, 0) False parent [0, 0, 0, 0, 4, 4] groups 2
(2, 5) True parent [0, 0, 0, 0, 0, 4] groups 1


In [ ]:
def find_provinces(M):
    n = len(M)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    groups = n
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j] == 1:
                ri, rj = find(i), find(j)
                if ri != rj:
                    parent[rj] = ri
                    groups -= 1
    return groups

assert find_provinces([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_provinces([[1, 0], [0, 1]]) == 2
assert find_provinces([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) == 1
print(find_provinces([[1, 1, 0], [1, 1, 0], [0, 0, 1]]), find_provinces([[1, 0], [0, 1]]))

2 2


In [ ]:
def find_redundant(edges):
    parent = list(range(len(edges) + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]
        parent[rb] = ra
    return []

assert find_redundant([(1, 2), (1, 3), (2, 3)]) == [2, 3]
assert find_redundant([(1, 2), (2, 3), (3, 4), (1, 4), (1, 5)]) == [1, 4]
print(find_redundant([(1, 2), (1, 3), (2, 3)]), find_redundant([(1, 2), (2, 3), (3, 4), (1, 4), (1, 5)]))

[2, 3] [1, 4]


In [ ]:
def num_islands_online(m, n, positions):
    parent = {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count, answer = 0, []
    for r, c in positions:
        cell = r * n + c
        if cell in parent:              # duplicate addition
            answer.append(count)
            continue
        parent[cell] = cell
        count += 1                      # a new island, maybe merged below
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            nb = nr * n + nc
            if 0 <= nr < m and 0 <= nc < n and nb in parent:
                ra, rb = find(cell), find(nb)
                if ra != rb:
                    parent[rb] = ra
                    count -= 1          # two islands became one
        answer.append(count)
    return answer

res = num_islands_online(3, 3, [(0, 0), (0, 1), (1, 2), (2, 1), (1, 1)])
assert res == [1, 1, 2, 3, 1]
print(res)
print(num_islands_online(1, 3, [(0, 0), (0, 0), (0, 2), (0, 1)]))

[1, 1, 2, 3, 1]
[1, 1, 2, 1]


## 8.7 Dijkstra: Shortest Paths with Weights

In [ ]:
import heapq

def dijkstra(graph, source):
    dist = {v: float("inf") for v in graph}
    dist[source] = 0
    heap = [(0, source)]                # (distance so far, node)
    while heap:
        d, node = heapq.heappop(heap)   # closest unsettled node
        if d > dist[node]:
            continue                    # stale entry: a shorter one was processed already
        for nxt, w in graph[node]:
            nd = d + w
            if nd < dist[nxt]:          # found a shorter route to nxt
                dist[nxt] = nd
                heapq.heappush(heap, (nd, nxt))
    return dist

graph = {
    0: [(1, 7), (2, 9), (5, 14)],
    1: [(0, 7), (2, 10), (3, 15)],
    2: [(0, 9), (1, 10), (3, 11), (5, 2)],
    3: [(1, 15), (2, 11), (4, 6)],
    4: [(3, 6), (5, 9)],
    5: [(0, 14), (2, 2), (4, 9)],
}
print(dijkstra(graph, 0))

{0: 0, 1: 7, 2: 9, 3: 20, 4: 20, 5: 11}


In [ ]:
import heapq
from collections import defaultdict

def network_delay(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {}
    heap = [(0, k)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in dist:                # already settled with a smaller distance
            continue
        dist[node] = d
        for nxt, w in graph[node]:
            if nxt not in dist:
                heapq.heappush(heap, (d + w, nxt))
    return max(dist.values()) if len(dist) == n else -1

assert network_delay([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay([[1, 2, 1]], 2, 1) == 1
assert network_delay([[1, 2, 1]], 2, 2) == -1
print(network_delay([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2), network_delay([[1, 2, 1]], 2, 1), network_delay([[1, 2, 1]], 2, 2))

2 1 -1


In [ ]:
import heapq

def minimum_effort(heights):
    rows, cols = len(heights), len(heights[0])
    best = [[float("inf")] * cols for _ in range(rows)]
    best[0][0] = 0
    heap = [(0, 0, 0)]                  # (effort so far, row, col)
    while heap:
        effort, r, c = heapq.heappop(heap)
        if effort > best[r][c]:
            continue
        if (r, c) == (rows - 1, cols - 1):
            return effort
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                step = abs(heights[nr][nc] - heights[r][c])
                ne = max(effort, step)  # the worst step so far, not the sum
                if ne < best[nr][nc]:
                    best[nr][nc] = ne
                    heapq.heappush(heap, (ne, nr, nc))

assert minimum_effort([[1, 2, 2], [3, 8, 2], [5, 3, 5]]) == 2
assert minimum_effort([[1, 2, 3], [3, 8, 4], [5, 3, 5]]) == 1
assert minimum_effort([[7]]) == 0
print(minimum_effort([[1, 2, 2], [3, 8, 2], [5, 3, 5]]), minimum_effort([[1, 2, 3], [3, 8, 4], [5, 3, 5]]), minimum_effort([[7]]))

2 1 0


In [ ]:
import heapq

def dijkstra_settled(graph, source):    # settles each node once, like the textbook
    dist, done = {source: 0}, set()
    heap = [(0, source)]
    while heap:
        d, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        for nxt, w in graph[node]:
            if nxt not in done and d + w < dist.get(nxt, float("inf")):
                dist[nxt] = d + w
                heapq.heappush(heap, (d + w, nxt))
    return dist

graph = {0: [(1, 2), (2, 5)], 1: [], 2: [(1, -4)]}
print(dijkstra_settled(graph, 0))       # true answer for node 1 is 1

{0: 0, 1: 2, 2: 5}


## 8.8 Bellman-Ford and Floyd-Warshall (Awareness)

In [ ]:
def bellman_ford(n, edges, source):
    dist = [float("inf")] * n
    dist[source] = 0
    for pass_no in range(1, n):         # V - 1 passes
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
        print("pass", pass_no, dist)
    for u, v, w in edges:               # one more pass: any improvement means a negative cycle
        if dist[u] + w < dist[v]:
            return None
    return dist

edges = [(2, 3, 2), (1, 2, -3), (0, 2, 5), (0, 1, 4)]   # edges deliberately in a bad order
print(bellman_ford(4, edges, 0))
print(bellman_ford(3, [(0, 1, 1), (1, 2, -3), (2, 1, 1)], 0))   # 1 to 2 to 1 costs -2 per loop

pass 1 [0, 4, 5, inf]
pass 2 [0, 4, 1, 7]
pass 3 [0, 4, 1, 3]
[0, 4, 1, 3]
pass 1 [0, -1, -2]
pass 2 [0, -3, -4]
None


In [ ]:
def cheapest_flight(n, flights, src, dst, k):
    dist = [float("inf")] * n
    dist[src] = 0
    for _ in range(k + 1):              # k stops means k + 1 flights
        nxt = dist[:]                   # read old values, write new ones
        for u, v, price in flights:
            if dist[u] + price < nxt[v]:
                nxt[v] = dist[u] + price
        dist = nxt
    return dist[dst] if dist[dst] != float("inf") else -1

flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
assert cheapest_flight(4, flights, 0, 3, 1) == 700
assert cheapest_flight(4, flights, 0, 3, 2) == 400
assert cheapest_flight(4, flights, 0, 3, 0) == -1
print(cheapest_flight(4, flights, 0, 3, 1), cheapest_flight(4, flights, 0, 3, 2), cheapest_flight(4, flights, 0, 3, 0))

700 400 -1


In [ ]:
INF = float("inf")

def floyd_warshall(d):
    n = len(d)
    d = [row[:] for row in d]
    for k in range(n):                  # allowed middle city: 0..k
        for i in range(n):
            for j in range(n):
                if d[i][k] + d[k][j] < d[i][j]:
                    d[i][j] = d[i][k] + d[k][j]
    return d

d = [[0, 3, INF, 7],
     [8, 0, 2, INF],
     [5, INF, 0, 1],
     [2, INF, INF, 0]]
for row in floyd_warshall(d):
    print(row)

[0, 3, 5, 6]
[5, 0, 2, 3]
[3, 6, 0, 1]
[2, 5, 7, 0]


## 8.9 Minimum Spanning Tree: Kruskal and Prim

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

def kruskal(n, edges):                  # edges are (a, b, weight)
    dsu, total, chosen = DSU(n), 0, []
    for a, b, w in sorted(edges, key=lambda e: e[2]):
        if dsu.union(a, b):             # joins two groups: keep it
            total += w
            chosen.append((a, b, w))
            if len(chosen) == n - 1:
                break
    return total, chosen

edges = [(0, 1, 7), (0, 2, 9), (0, 5, 14), (1, 2, 10), (1, 3, 15),
         (2, 3, 11), (2, 5, 2), (3, 4, 6), (4, 5, 9)]
total, chosen = kruskal(6, edges)
print(total)
print(chosen)
assert total == 33

33
[(2, 5, 2), (3, 4, 6), (0, 1, 7), (0, 2, 9), (4, 5, 9)]


In [ ]:
import heapq

def prim(graph, start=0):
    seen = {start}
    heap = [(w, start, v) for v, w in graph[start]]
    heapq.heapify(heap)
    total, chosen = 0, []
    while heap and len(seen) < len(graph):
        w, u, v = heapq.heappop(heap)
        if v in seen:
            continue                    # would close a cycle
        seen.add(v)
        total += w
        chosen.append((u, v, w))
        for nxt, nw in graph[v]:
            if nxt not in seen:
                heapq.heappush(heap, (nw, v, nxt))
    return total, chosen

graph = {
    0: [(1, 7), (2, 9), (5, 14)],
    1: [(0, 7), (2, 10), (3, 15)],
    2: [(0, 9), (1, 10), (3, 11), (5, 2)],
    3: [(1, 15), (2, 11), (4, 6)],
    4: [(3, 6), (5, 9)],
    5: [(0, 14), (2, 2), (4, 9)],
}
total, chosen = prim(graph)
print(total)
print(chosen)
assert total == 33

33
[(0, 1, 7), (0, 2, 9), (2, 5, 2), (5, 4, 9), (4, 3, 6)]


In [ ]:
def min_cost_connect(points):
    n = len(points)
    INF = float("inf")
    best = [INF] * n                    # cheapest link from the tree to each point
    in_tree = [False] * n
    best[0] = 0
    total = 0
    for _ in range(n):
        u = min((i for i in range(n) if not in_tree[i]), key=lambda i: best[i])
        in_tree[u] = True
        total += best[u]
        for v in range(n):
            if not in_tree[v]:
                dist = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                best[v] = min(best[v], dist)
    return total

pts = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]
assert min_cost_connect(pts) == 20
assert min_cost_connect([[0, 0]]) == 0
print(min_cost_connect(pts), min_cost_connect([[0, 0]]), min_cost_connect([[1, 1], [1, 1]]))

20 0 0


## 8.10 Bipartite Check: Two Teams, No Enemies Together

In [ ]:
from collections import deque

def is_bipartite(graph):
    color = {}                          # node -> 0 or 1
    for start in range(len(graph)):     # graph may be disconnected
        if start in color:
            continue
        color[start] = 0
        queue = deque([start])
        while queue:
            node = queue.popleft()
            for nxt in graph[node]:
                if nxt not in color:
                    color[nxt] = 1 - color[node]    # force the opposite colour
                    queue.append(nxt)
                elif color[nxt] == color[node]:
                    return False                    # forced to be both
    return True

cycle6 = [[1, 5], [0, 2], [1, 3], [2, 4], [3, 5], [4, 0]]
cycle5 = [[1, 4], [0, 2], [1, 3], [2, 4], [3, 0]]
print(is_bipartite(cycle6), is_bipartite(cycle5))

True False


In [ ]:
from collections import deque

def possible_bipartition(n, dislikes):
    graph = [[] for _ in range(n + 1)]
    for a, b in dislikes:
        graph[a].append(b)
        graph[b].append(a)
    color = {}
    for s in range(1, n + 1):
        if s in color:
            continue
        color[s] = 0
        queue = deque([s])
        while queue:
            u = queue.popleft()
            for v in graph[u]:
                if v not in color:
                    color[v] = 1 - color[u]
                    queue.append(v)
                elif color[v] == color[u]:
                    return False
    return True

assert possible_bipartition(4, [[1, 2], [1, 3], [2, 4]]) is True
assert possible_bipartition(3, [[1, 2], [1, 3], [2, 3]]) is False
assert possible_bipartition(5, []) is True
print(possible_bipartition(4, [[1, 2], [1, 3], [2, 4]]), possible_bipartition(3, [[1, 2], [1, 3], [2, 3]]), possible_bipartition(5, []))

True False True


## 8.11 How to Choose the Algorithm

Say these questions aloud in an interview. They take 30 seconds and name the algorithm for you.

## Chapter recap

## Practice set

Twenty-two problems, from easy to hard. Try each one on paper first, then read the answer. Every code answer was run and the output is real. Problem statements are written fresh for this book.

In [ ]:
from collections import deque

graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]
dist = {5: 0}
order = []
queue = deque([5])
while queue:
    node = queue.popleft()
    order.append(node)
    for nxt in graph[node]:
        if nxt not in dist:
            dist[nxt] = dist[node] + 1
            queue.append(nxt)
print(order)
print(dist[0])

In [ ]:
graph = [[1, 2], [0, 3], [0, 3, 4], [1, 2, 5], [2, 5], [3, 4]]

def dfs(node, seen, order):
    seen.add(node)
    order.append(node)
    for nxt in graph[node]:
        if nxt not in seen:
            dfs(nxt, seen, order)

order = []
dfs(3, set(), order)
print(order)

In [ ]:
from collections import deque

n = 6
graph = [[j for j in range(n) if j != i] for i in range(n)]
visited = set()
queue = deque([0])
pushes = 1
while queue:
    node = queue.popleft()
    if node in visited:
        continue
    visited.add(node)                   # marked when popped
    for nxt in graph[node]:
        if nxt not in visited:
            queue.append(nxt)
            pushes += 1
print(len(visited), pushes)

In [ ]:
from collections import deque

n = 6
graph = [[j for j in range(n) if j != i] for i in range(n)]
visited = {0}
queue = deque([0])
pushes = 1
while queue:
    node = queue.popleft()
    for nxt in graph[node]:
        if nxt not in visited:
            visited.add(nxt)            # marked when pushed
            queue.append(nxt)
            pushes += 1
print(len(visited), pushes)

6 6


In [ ]:
edges = [(0, 1), (1, 2), (2, 3)]
graph = {i: [] for i in range(4)}
for a, b in edges:
    graph[a].append(b)                  # only one direction stored

def reachable(start):
    seen, stack = {start}, [start]
    while stack:
        node = stack.pop()
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                stack.append(nxt)
    return sorted(seen)

print(reachable(0))
print(reachable(3))

In [ ]:
parent = list(range(6))

def find(x):
    steps = 0
    while parent[x] != x:
        x = parent[x]
        steps += 1
    return x, steps

for a in range(5):                      # union(0,1), union(1,2), ...
    ra, _ = find(a)
    rb, _ = find(a + 1)
    parent[ra] = rb                     # always hang the first root under the second
print(parent)
print(find(0))

In [ ]:
def count_islands(grid, dirs):
    rows, cols = len(grid), len(grid[0])
    seen = set()
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in seen:
                count += 1
                stack = [(r, c)]
                seen.add((r, c))
                while stack:
                    cr, cc = stack.pop()
                    for dr, dc in dirs:
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1" and (nr, nc) not in seen:
                            seen.add((nr, nc))
                            stack.append((nr, nc))
    return count

grid = ["1100", "0010", "0001"]
four = [(-1, 0), (1, 0), (0, -1), (0, 1)]
eight = four + [(-1, -1), (-1, 1), (1, -1), (1, 1)]
print(count_islands(grid, four), count_islands(grid, eight))

In [ ]:
import heapq

graph = {0: [(1, 4), (2, 1)], 1: [], 2: [(1, 2)]}
dist = {}
heap = [(0, 0)]                         # pushed as (node, distance)
while heap:
    node, d = heapq.heappop(heap)
    if node in dist:
        continue
    dist[node] = d
    for nxt, w in graph[node]:
        heapq.heappush(heap, (nxt, d + w))
print(dist)

In [ ]:
def has_cycle(n, edges):
    graph = [[] for _ in range(n)]
    for a, b in edges:
        graph[a].append(b)
    seen = set()

    def dfs(node):
        seen.add(node)
        for nxt in graph[node]:
            if nxt in seen or dfs(nxt):
                return True
        return False

    return any(dfs(s) for s in range(n) if s not in seen)

print(has_cycle(3, [(0, 1), (0, 2), (1, 2)]))

In [ ]:
from collections import deque

def knight_moves(size, start, goal):
    jumps = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
    dist = {start: 0}
    queue = deque([start])
    while queue:
        r, c = queue.popleft()
        if (r, c) == goal:
            return dist[(r, c)]
        for dr, dc in jumps:
            nr, nc = r + dr, c + dc
            if 0 <= nr < size and 0 <= nc < size and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                queue.append((nr, nc))
    return -1

print(knight_moves(5, (0, 0), (4, 4)))
print(knight_moves(8, (0, 0), (7, 7)))
print(knight_moves(3, (0, 0), (1, 1)))
assert knight_moves(5, (0, 0), (0, 0)) == 0

4
6
-1


In [ ]:
from collections import deque

def shortest_with_breaks(grid, k):
    rows, cols = len(grid), len(grid[0])
    start = (0, 0, 0)
    seen = {start}
    queue = deque([(0, 0, 0, 0)])       # row, col, walls broken, steps
    while queue:
        r, c, used, steps = queue.popleft()
        if (r, c) == (rows - 1, cols - 1):
            return steps
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nu = used + grid[nr][nc]
                if nu <= k and (nr, nc, nu) not in seen:
                    seen.add((nr, nc, nu))
                    queue.append((nr, nc, nu, steps + 1))
    return -1

grid = [[0, 0, 0],
        [1, 1, 0],
        [0, 0, 0],
        [0, 1, 1],
        [0, 0, 0]]
print(shortest_with_breaks(grid, 0), shortest_with_breaks(grid, 1))
assert shortest_with_breaks([[0]], 0) == 0

10 6


In [ ]:
import math

def has_arbitrage(n, rates):            # rates: (from, to, rate)
    edges = [(a, b, -math.log(r)) for a, b, r in rates]
    dist = [0.0] * n                    # start every currency at 0
    for _ in range(n):
        changed = False
        for a, b, w in edges:
            if dist[a] + w < dist[b] - 1e-12:
                dist[b] = dist[a] + w
                changed = True
        if not changed:
            return False                # settled: no negative cycle
    return True                         # still improving after n passes

profit = [(0, 1, 0.9), (1, 2, 0.9), (2, 0, 1.3)]     # 0.9 * 0.9 * 1.3 = 1.053
loss = [(0, 1, 0.9), (1, 2, 0.9), (2, 0, 1.2)]       # 0.9 * 0.9 * 1.2 = 0.972
print(has_arbitrage(3, profit), has_arbitrage(3, loss))

True False


In [ ]:
def merge_accounts(accounts):
    parent = {}
    owner = {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for name, *emails in accounts:
        for e in emails:
            parent.setdefault(e, e)
            owner[e] = name
        for e in emails[1:]:
            parent[find(e)] = find(emails[0])      # join to the first email
    groups = {}
    for e in parent:
        groups.setdefault(find(e), []).append(e)
    return sorted([owner[root]] + sorted(es) for root, es in groups.items())

accounts = [["John", "a@x.com", "b@x.com"],
            ["John", "b@x.com", "c@x.com"],
            ["Mary", "m@x.com"],
            ["John", "z@x.com"]]
for row in merge_accounts(accounts):
    print(row)

['John', 'a@x.com', 'b@x.com', 'c@x.com']
['John', 'z@x.com']
['Mary', 'm@x.com']


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch08.html